In [1]:
import pandas as pd
from pathlib import Path
import altair as alt

In [2]:
df = pd.read_csv('website/cleaned_listings.csv')

In [3]:
# filter the data to exclude outliers
Q1 = df['minimum_nights'].quantile(0.25)
Q3 = df['minimum_nights'].quantile(0.75)
IQR = Q3 - Q1

threshold = Q3 + 1.5 * IQR
df_filtered2 = df[(df['minimum_nights'] <= threshold) & (df['minimum_nights'] > 15)].copy()
df_filtered1 = df[df['minimum_nights'] <= 15].copy()

# --- 変更箇所: 四分位数に基づいた動的なカテゴリ作成 ---

# 全データの分布に基づいて四分位数を計算し、境界値(bin_edges)を取得
# duplicates='drop' は、データに0が多い場合などで境界値が重複してエラーになるのを防ぎます
_, bin_edges = pd.qcut(df['availability_365'], q=4, retbins=True, duplicates='drop')

# 境界値に基づいてラベルを動的に作成
categorical_order = []
# 4分割きれいにできた場合用の接頭辞（区切りが減った場合は使わないロジックにします）
prefixes = ['Low', 'Medium', 'High', 'Very High']

for i in range(len(bin_edges)-1):
    lower = int(bin_edges[i])
    upper = int(bin_edges[i+1])
    
    # 4分割であれば Low/Medium などを付け、そうでなければ数値範囲のみ表示
    if len(bin_edges)-1 == 4:
        label = f"{prefixes[i]} ({lower}-{upper})"
    else:
        label = f"{lower}-{upper}"
    categorical_order.append(label)

# 作成したbinsとラベルを適用
df_filtered1['availability_category'] = pd.cut(
    df_filtered1['availability_365'], 
    bins=bin_edges,
    labels=categorical_order,
    include_lowest=True # 最小値(0)を含める
)

In [4]:
color_order = ['null'] + categorical_order

order_map = {cat: i for i, cat in enumerate(color_order)}

df_filtered1 = df_filtered1.copy()
df_filtered1['availability_order'] = df_filtered1['availability_category'].map(order_map)

In [5]:
color_order = ['null'] + categorical_order

chart1 = alt.Chart(df_filtered1).mark_bar(opacity=0.7,  stroke=None   ).encode(
    x=alt.X('minimum_nights:Q', 
            bin=alt.Bin(maxbins=20),
            title='Minimum Nights Required'),
    y=alt.Y('count()', title='Number of Listings'),
    color=alt.Color('availability_category:N',
                    title='Availability (365 days)',
                    scale=alt.Scale(scheme='viridis', domain=color_order), sort=color_order),
    order=alt.Order('availability_order:Q'),
    tooltip=[
        alt.Tooltip('minimum_nights:Q', bin=True, title='Minimum Nights'),
        alt.Tooltip('count()', title='Count'),
        alt.Tooltip('availability_category:N', title='Availability Level')
    ]
).properties(
    width=700,
    height=400,
    title='Distribution of Minimum Nights by Availability Level'
).interactive()

c1_json = chart1.to_json()

with open('website/t4-barplot1_spec.json', 'w') as f:
    f.write(c1_json)



In [6]:
# Cell 15で計算した bin_edges と categorical_order を再利用して一貫性を保つ
df_filtered2['availability_category'] = pd.cut(
    df_filtered2['availability_365'], 
    bins=bin_edges,             # 共通の境界値を使用
    labels=categorical_order,   # 共通のラベルを使用
    include_lowest=True
)

df_filtered2 = df_filtered2.copy()
df_filtered2['availability_order'] = df_filtered2['availability_category'].map(order_map)

# チャート作成部分は変更なし（color_order等は自動的に更新されたcategorical_orderを参照します）
chart2 = alt.Chart(df_filtered2).mark_bar(opacity=0.7,  stroke=None).encode(
    x=alt.X('minimum_nights:Q', 
            bin=alt.Bin(maxbins=20),
            title='Minimum Nights Required'),
    y=alt.Y('count()', title='Number of Listings'),
    color=alt.Color('availability_category:N',
                    title='Availability (365 days)',
                    scale=alt.Scale(scheme='viridis', domain=color_order), sort=color_order),
    order=alt.Order('availability_order:Q'),
    tooltip=[
        alt.Tooltip('minimum_nights:Q', bin=True, title='Minimum Nights'),
        alt.Tooltip('count()', title='Count'),
        alt.Tooltip('availability_category:N', title='Availability Level')
    ]
).properties(
    width=800,
    height=600,
    title='Distribution of Minimum Nights by Availability Level'
).interactive()

c2_json = chart2.to_json()

with open('website/t4-barplot2_spec.json', 'w') as f:
    f.write(c2_json)

chart2

alt.Chart(...)